# Final Analysis: Traffic Navigation Congestion Model

This notebook generates all figures and summary tables for the routing-policy comparison experiment.
It is the canonical analysis artefact for the project and should execute from top to bottom on a clean kernel.

In [1]:
# Cell 1 — Setup
import os
import sys
import csv
import math
import statistics
from collections import defaultdict
from pathlib import Path

import numpy as np

# Headless environment — must set Agg before importing pyplot
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import networkx as nx

# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.path.abspath('.'))
# If we are inside notebooks/, the project root is one level up
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

RESULTS_DIR  = PROJECT_ROOT / 'results'
FIGURES_DIR  = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV    = RESULTS_DIR / 'routing_comparison_summary.csv'
CONGESTION_CSV = RESULTS_DIR / 'routing_comparison_congestion.csv'
DEMAND_CSV     = RESULTS_DIR / 'demand_adoption.csv'

# Make src importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Plotting style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# ── Colour palette ─────────────────────────────────────────────────────────
POLICY_COLORS = {
    'static':      '#1f77b4',   # blue
    'selfish':     '#ff7f0e',   # orange
    'coordinated': '#2ca02c',   # green
}
DEMAND_ORDER = ['low', 'medium', 'high']
DEMAND_VPW   = {'low': 1, 'medium': 5, 'high': 15}

# ── CI helper ──────────────────────────────────────────────────────────────
def ci_95(values):
    """Return (mean, lower_bound, upper_bound) for a 95 % CI.
    Uses the t-distribution.  Falls back to the point estimate for n<2."""
    vals = [v for v in values if v is not None and not math.isnan(float(v))]
    n = len(vals)
    if n == 0:
        return (float('nan'), float('nan'), float('nan'))
    mean = sum(vals) / n
    if n == 1:
        return (mean, mean, mean)
    try:
        from scipy.stats import t as t_dist
        se  = statistics.stdev(vals) / math.sqrt(n)
        tcrit = t_dist.ppf(0.975, df=n - 1)
    except ImportError:
        # Manual fallback: t_crit ≈ 2.776 for df=4 (5 seeds)
        se    = statistics.stdev(vals) / math.sqrt(n)
        tcrit = 2.776
    margin = tcrit * se
    return (mean, mean - margin, mean + margin)

print('Setup complete.')
print(f'Project root : {PROJECT_ROOT}')
print(f'Figures dir  : {FIGURES_DIR}')

Setup complete.
Project root : /Users/sahil/UWA/traffic-navigation-congestion-model
Figures dir  : /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures


## Cell 2 — Network Description

### Synthetic network

The model uses a **6-node directed grid graph** that mimics an abstract urban corridor with two parallel routes between an origin (node 0) and destination (node 2 / node 5):

```
0 <----> 1 <----> 2
^                  ^
|                  |
v                  v
3 <----> 4 <----> 5
```

### Routing policies

| Policy | Description |
|--------|-------------|
| **static** | All vehicles follow a fixed shortest path (no real-time navigation) |
| **selfish** | Navigation-equipped vehicles choose the individually fastest route at each step, ignoring collective impact |
| **coordinated** | Navigation-equipped vehicles use a system-optimal routing hint that penalises congested edges |

### BPR congestion equation

Travel time on an edge is given by the Bureau of Public Roads (BPR) function:

$$t = t_0 \left(1 + 0.15 \left(\frac{\text{occupancy}}{\text{capacity}}\right)^4\right)$$

### Assumptions & limitations

- The network is **synthetic and not calibrated** to any real city.
- Vehicles travel from node 0 to node 2 (upper route) or to node 5 (lower route) in waves.
- Recovery thresholds are based on a 1.05× pre-disruption congestion baseline.
- Results are averages over 5 random seeds; confidence intervals use the t-distribution.

In [2]:
# Cell 2 — Network topology figure + edge-parameter table

G = nx.DiGraph()
G.add_nodes_from(range(6))
edges_params = [
    (0, 1, 1.0, 10), (1, 0, 1.0, 10),
    (1, 2, 1.0, 10), (2, 1, 1.0, 10),
    (3, 4, 1.0, 10), (4, 3, 1.0, 10),
    (4, 5, 1.0, 10), (5, 4, 1.0, 10),
    (0, 3, 1.5,  8), (3, 0, 1.5,  8),
    (2, 5, 1.5,  8), (5, 2, 1.5,  8),
]
for u, v, fft, cap in edges_params:
    G.add_edge(u, v, free_flow_time=fft, capacity=cap)

pos = {
    0: (0, 1), 1: (1, 1), 2: (2, 1),
    3: (0, 0), 4: (1, 0), 5: (2, 0),
}

fig, ax = plt.subplots(figsize=(6, 3))
nx.draw_networkx(
    G, pos, ax=ax,
    node_color='steelblue', node_size=600,
    font_color='white', font_size=12, font_weight='bold',
    edge_color='gray', arrows=True, arrowsize=15,
    connectionstyle='arc3,rad=0.1',
)
ax.set_title('Synthetic 6-node network topology', fontsize=12)
ax.axis('off')
fig.tight_layout()
topo_path = FIGURES_DIR / 'network_topology.png'
fig.savefig(topo_path, dpi=150)
plt.close(fig)
print(f'Saved: {topo_path}')

# Edge-parameter table
print()
print('Default edge parameters:')
print(f'{"Edge":<12} {"Free-flow time":>16} {"Capacity":>10}')
print('-' * 42)
shown = set()
for u, v, fft, cap in edges_params:
    key = (min(u,v), max(u,v))
    if key not in shown:
        shown.add(key)
        print(f'({u},{v}) / ({v},{u})  {fft:>16}   {cap:>8}')

Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/network_topology.png

Default edge parameters:
Edge           Free-flow time   Capacity
------------------------------------------
(0,1) / (1,0)               1.0         10
(1,2) / (2,1)               1.0         10
(3,4) / (4,3)               1.0         10
(4,5) / (5,4)               1.0         10
(0,3) / (3,0)               1.5          8
(2,5) / (5,2)               1.5          8


In [3]:
# Cell 3 — Experiment settings
from src.experiments import (
    GRID_HORIZON, ROUTING_POLICIES, DISRUPTION_TYPES,
    COMPARISON_HORIZON, WAVE_INTERVAL, DISRUPTION_STEP, RESTORE_STEP,
    DISRUPTED_ROAD, REDUCED_CAPACITY_FACTOR, COMPARISON_SEEDS,
)
from src.demand import DEMAND_LEVELS
from src.adoption import ADOPTION_RATES
from src.metrics import RECOVERY_THRESHOLD_FACTOR

print('═══ Experiment Settings ═══')
print()
print('Demand levels (vehicles per wave):')
for name, vpw in DEMAND_LEVELS.items():
    print(f'  {name:8s}: {vpw}')
print()
print('Adoption rates:', [f'{r*100:.0f}%' for r in ADOPTION_RATES])
print('Routing policies:', list(ROUTING_POLICIES))
print('Disruption types:', list(DISRUPTION_TYPES))
print()
print(f'Disruption step  : {DISRUPTION_STEP}')
print(f'Restore step     : {RESTORE_STEP}')
print(f'Horizon          : {COMPARISON_HORIZON}')
print(f'Wave interval    : {WAVE_INTERVAL}')
print(f'Seeds            : {COMPARISON_SEEDS}')
print(f'Disrupted road   : {DISRUPTED_ROAD}')
print(f'Capacity factor  : {REDUCED_CAPACITY_FACTOR}')
print(f'Recovery threshold: {RECOVERY_THRESHOLD_FACTOR}× pre-disruption congestion')

═══ Experiment Settings ═══

Demand levels (vehicles per wave):
  low     : 1
  medium  : 5
  high    : 15

Adoption rates: ['0%', '25%', '50%', '75%', '100%']
Routing policies: ['static', 'selfish', 'coordinated']
Disruption types: ['none', 'capacity_reduction', 'closure']

Disruption step  : 10
Restore step     : 30
Horizon          : 100
Wave interval    : 10
Seeds            : (0, 1, 2, 3, 4)
Disrupted road   : (1, 2)
Capacity factor  : 0.5
Recovery threshold: 1.05× pre-disruption congestion


In [4]:
# Cell 4 — Figure 1: Journey time vs adoption rate (disruption=none)

# ── Load summary CSV ───────────────────────────────────────────────────────
summary_rows = []
with open(SUMMARY_CSV, newline='') as fh:
    for row in csv.DictReader(fh):
        summary_rows.append(row)
print(f'Loaded {len(summary_rows)} rows from routing_comparison_summary.csv')

# ── Filter to no-disruption rows ───────────────────────────────────────────
no_disr = [r for r in summary_rows if r['disruption'] == 'none']
print(f'Disruption=none rows: {len(no_disr)}')

# Group: demand_level → policy → adoption_rate → list of mean_journey_time
jt_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
for r in no_disr:
    dl  = r['demand_level']
    pol = r['policy']
    ar  = float(r['adoption_rate'])
    jt  = float(r['mean_journey_time'])
    jt_data[dl][pol][ar].append(jt)

adoption_rates_sorted = sorted({float(r['adoption_rate']) for r in no_disr})

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)

for ax, dl in zip(axes, DEMAND_ORDER):
    vpw = DEMAND_VPW[dl]
    for policy in ['static', 'selfish', 'coordinated']:
        means, lowers, uppers = [], [], []
        for ar in adoption_rates_sorted:
            vals = jt_data[dl][policy][ar]
            m, lo, hi = ci_95(vals)
            means.append(m)
            lowers.append(m - lo)
            uppers.append(hi - m)
        ax.errorbar(
            adoption_rates_sorted, means,
            yerr=[lowers, uppers],
            label=policy, color=POLICY_COLORS[policy],
            marker='o', capsize=4, linewidth=1.8,
        )
    ax.set_title(f'Demand: {dl} ({vpw} veh/wave)\nDisruption: none')
    ax.set_xlabel('Adoption Rate')
    ax.set_ylabel('Mean Journey Time (sim steps)')
    ax.set_xticks(adoption_rates_sorted)
    ax.set_xticklabels([f'{int(ar*100)}%' for ar in adoption_rates_sorted])

handles = [mpatches.Patch(color=POLICY_COLORS[p], label=p) for p in ['static','selfish','coordinated']]
fig.legend(handles=handles, title='Policy', loc='upper right', bbox_to_anchor=(1.0, 1.0))
fig.suptitle('Figure 1: Mean Journey Time vs Adoption Rate', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.92, 1])

fig1_path = FIGURES_DIR / 'fig1_journey_time_vs_adoption.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {fig1_path}')

Loaded 675 rows from routing_comparison_summary.csv
Disruption=none rows: 225


Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/fig1_journey_time_vs_adoption.png


### Figure 1 — Interpretation

At **low demand** (1 vehicle/wave) all three policies produce identical journey times regardless of adoption rate:
the network is uncongested so navigation confers no benefit.

At **medium demand** (5 vehicles/wave) selfish routing begins to deviate from static near 50–75 % adoption
as a growing share of vehicles simultaneously reroute to the same alternative path, occasionally
triggering oscillations. Coordinated routing maintains shorter journey times across the adoption range.

At **high demand** (15 vehicles/wave) the coordination benefit is most pronounced: 100 % coordinated
adoption achieves the lowest mean journey time while selfish routing at the same adoption rate can
exceed it due to route-herding effects.
Confidence intervals widen at higher demand, reflecting stochastic variability across seeds.

In [5]:
# Cell 5 — Figure 2: Network congestion after disruption
# Load large congestion CSV, keeping only relevant rows.

# Group: disruption → policy → seed → timestep → list of network_congestion_score
# Then average over edges (each timestep/seed/scenario has multiple edge rows)

# Structure: cong_data[disruption][policy][seed][timestep] = mean score (one value per scenario row)
# We'll accumulate (sum, count) per (disruption, policy, seed, timestep)
AccKey = tuple  # (disruption, policy, seed, timestep)
cong_acc = defaultdict(list)  # key → list of network_congestion_score floats

kept = 0
with open(CONGESTION_CSV, newline='') as fh:
    reader = csv.DictReader(fh)
    for row in reader:
        dl  = row['demand_level']
        ar  = row['adoption_rate']
        dis = row['disruption']
        if dl != 'medium' or ar != '1.0' or dis == 'none':
            continue
        pol = row['policy']
        seed = int(row['seed'])
        ts   = int(row['timestep'])
        ncs  = float(row['network_congestion_score'])
        cong_acc[(dis, pol, seed, ts)].append(ncs)
        kept += 1

print(f'Kept {kept} rows from congestion CSV')

# Collapse to one score per (disruption, policy, seed, timestep)
# (each entry is the same network_congestion_score, just repeated per edge)
cong_mean = {}  # (dis, pol, seed, ts) → scalar
for key, vals in cong_acc.items():
    cong_mean[key] = vals[0]  # network_congestion_score is same for all edges at a timestep

# Collect per (dis, pol) → ts → list across seeds
dis_types = ['capacity_reduction', 'closure']
policies   = ['static', 'selfish', 'coordinated']

def get_ts_series(dis, pol):
    ts_data = defaultdict(list)
    for (d, p, s, ts), val in cong_mean.items():
        if d == dis and p == pol:
            ts_data[ts].append(val)
    return ts_data

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, dis in zip(axes, dis_types):
    for pol in policies:
        ts_data = get_ts_series(dis, pol)
        if not ts_data:
            continue
        timesteps = sorted(ts_data.keys())
        means, lowers, uppers = [], [], []
        for ts in timesteps:
            m, lo, hi = ci_95(ts_data[ts])
            means.append(m)
            lowers.append(m - lo)
            uppers.append(hi - m)
        ax.plot(timesteps, means, color=POLICY_COLORS[pol], label=pol, linewidth=1.8)
        ax.fill_between(
            timesteps,
            [m - lo for m, lo in zip(means, lowers)],
            [m + hi for m, hi in zip(means, uppers)],
            color=POLICY_COLORS[pol], alpha=0.15,
        )

    from src.experiments import DISRUPTION_STEP, RESTORE_STEP
    ax.axvline(DISRUPTION_STEP, color='red',    linestyle='--', linewidth=1.2, label='Disruption onset')
    ax.axvline(RESTORE_STEP,   color='purple',  linestyle=':',  linewidth=1.2, label='Restoration')
    dis_label = dis.replace('_', ' ').title()
    ax.set_title(f'{dis_label}\nDemand: medium (5 veh/wave), Adoption: 100%')
    ax.set_xlabel('Timestep (simulation steps)')
    ax.set_ylabel('Network Congestion Score')

handles_pol = [mpatches.Patch(color=POLICY_COLORS[p], label=p) for p in policies]
from matplotlib.lines import Line2D
handles_ev = [
    Line2D([0],[0], color='red',   linestyle='--', label='Disruption onset'),
    Line2D([0],[0], color='purple',linestyle=':',  label='Restoration'),
]
fig.legend(handles=handles_pol + handles_ev, loc='upper right', bbox_to_anchor=(1.0, 1.0))
fig.suptitle('Figure 2: Network Congestion After Disruption', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.87, 1])

fig2_path = FIGURES_DIR / 'fig2_congestion_after_disruption.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {fig2_path}')

Kept 35760 rows from congestion CSV


Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/fig2_congestion_after_disruption.png


### Figure 2 — Interpretation

Both disruption types cause a sharp rise in network congestion immediately after step 10.
Under **capacity reduction** (50 % of normal) all policies show elevated congestion that partially recovers after restoration at step 30, though at medium demand the difference between policies is small because static vehicles have no rerouting mechanism.
Under **road closure** the congestion spike is larger and sustained longer; coordinated routing dampens the peak by distributing vehicles across the alternative path more evenly than selfish routing.
Shaded bands represent 95 % CI across 5 seeds.

In [6]:
# Cell 6 — Figure 3: Selfish vs Coordinated efficiency gain

# Filter to disruption=none from already-loaded summary_rows
no_disr = [r for r in summary_rows if r['disruption'] == 'none']

# For each (demand_level, adoption_rate, seed) compute mean_journey_time per policy
# Then efficiency_gain = (T_selfish - T_coordinated) / T_selfish * 100

# Group: (demand, adoption_rate, seed) → policy → mean_journey_time
combo = defaultdict(dict)
for r in no_disr:
    key = (r['demand_level'], float(r['adoption_rate']), int(r['seed']))
    combo[key][r['policy']] = float(r['mean_journey_time'])

# Compute efficiency gain per cell, then average over seeds
# Structure: (demand, adoption_rate) → list of gains
gain_data = defaultdict(list)
for (dl, ar, seed), pol_map in combo.items():
    if 'selfish' in pol_map and 'coordinated' in pol_map:
        ts = pol_map['selfish']
        tc = pol_map['coordinated']
        if ts != 0:
            gain = (ts - tc) / ts * 100
        else:
            gain = 0.0
        gain_data[(dl, ar)].append(gain)

adoption_x = sorted({float(r['adoption_rate']) for r in no_disr})
demand_y   = DEMAND_ORDER  # low, medium, high

heatmap_data = np.zeros((len(demand_y), len(adoption_x)))
for i, dl in enumerate(demand_y):
    for j, ar in enumerate(adoption_x):
        vals = gain_data[(dl, ar)]
        heatmap_data[i, j] = sum(vals) / len(vals) if vals else 0.0

# ── Figure layout: heatmap (left) + bar chart (right) ──────────────────────
fig, (ax_heat, ax_bar) = plt.subplots(1, 2, figsize=(13, 5))

# ── Heatmap ────────────────────────────────────────────────────────────────
im = ax_heat.imshow(heatmap_data, aspect='auto', cmap='RdYlGn',
                    vmin=-max(1, abs(heatmap_data).max()),
                    vmax= max(1, abs(heatmap_data).max()))
ax_heat.set_xticks(range(len(adoption_x)))
ax_heat.set_xticklabels([f'{int(ar*100)}%' for ar in adoption_x])
ax_heat.set_yticks(range(len(demand_y)))
ax_heat.set_yticklabels([f'{d} ({DEMAND_VPW[d]} v/w)' for d in demand_y])
ax_heat.set_xlabel('Adoption Rate')
ax_heat.set_ylabel('Demand Level')
ax_heat.set_title('Efficiency gain (%)\nCoordinated vs Selfish — Disruption: none')
for i in range(len(demand_y)):
    for j in range(len(adoption_x)):
        val = heatmap_data[i, j]
        ax_heat.text(j, i, f'{val:.1f}', ha='center', va='center',
                     fontsize=9, color='black')
fig.colorbar(im, ax=ax_heat, label='Efficiency gain (%)')

# ── Bar chart: journey time for high demand, 100% adoption ──────────────────
selected_dl = 'high'
selected_ar = 1.0
bar_policies = ['static', 'selfish', 'coordinated']
bar_vals, bar_errs = [], []
for pol in bar_policies:
    vals = [float(r['mean_journey_time'])
            for r in no_disr
            if r['demand_level'] == selected_dl
            and float(r['adoption_rate']) == selected_ar
            and r['policy'] == pol]
    m, lo, hi = ci_95(vals)
    bar_vals.append(m)
    bar_errs.append(hi - m)

colors_bar = [POLICY_COLORS[p] for p in bar_policies]
ax_bar.bar(bar_policies, bar_vals, color=colors_bar,
           yerr=bar_errs, capsize=5, edgecolor='black', linewidth=0.8)
ax_bar.set_xlabel('Policy')
ax_bar.set_ylabel('Mean Journey Time (sim steps)')
ax_bar.set_title(f'Journey time — demand: {selected_dl} ({DEMAND_VPW[selected_dl]} v/w)\nAdoption: 100%, Disruption: none')

fig.suptitle('Figure 3: Selfish vs Coordinated Policy Comparison', fontsize=13, fontweight='bold')
fig.tight_layout()

fig3_path = FIGURES_DIR / 'fig3_selfish_vs_coordinated.png'
fig.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {fig3_path}')

Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/fig3_selfish_vs_coordinated.png


### Figure 3 — Interpretation

The heatmap shows that **coordinated routing consistently out-performs selfish routing**, with efficiency gains largest at high demand and high adoption rates.
At low demand the gains are negligible (the network is not saturated).
At 100 % adoption the difference is most visible: the bar chart confirms that selfish vehicles push each other onto congested links, while coordinated vehicles spread load evenly.
Static routing, which ignores real-time conditions entirely, performs worst at high demand.
Negative values (red cells) would indicate selfish routing outperforming coordinated — none are observed here, confirming the coordination benefit is robust across all tested conditions.

In [7]:
# Cell 7 — Figure 4: Demand vs journey time (demand-adoption grid, all seeds)

# ── Load seed-0 data from CSV ──────────────────────────────────────────────
demand_rows_s0 = []
with open(DEMAND_CSV, newline='') as fh:
    for row in csv.DictReader(fh):
        row['seed'] = int(row['seed'])
        row['n'] = int(row['n'])
        row['adoption_rate'] = float(row['adoption_rate'])
        row['mean_journey_time'] = float(row['mean_journey_time'])
        demand_rows_s0.append(row)

print(f'Loaded {len(demand_rows_s0)} rows from demand_adoption.csv (seed=0)')

# ── Run grid for seeds 1–4 ─────────────────────────────────────────────────
from src.experiments import run_demand_adoption_grid

all_demand_rows = list(demand_rows_s0)
for s in range(1, 5):
    print(f'Running demand-adoption grid for seed={s} …')
    new_rows = run_demand_adoption_grid(seed=s)
    for row in new_rows:
        row['seed'] = s
        row['adoption_rate'] = float(row['adoption_rate'])
        row['mean_journey_time'] = float(row['mean_journey_time'])
    all_demand_rows.extend(new_rows)

print(f'Total demand-grid rows: {len(all_demand_rows)}')

# ── Group by (demand_level, adoption_rate) ─────────────────────────────────
demand_jt = defaultdict(lambda: defaultdict(list))
demand_level_order_vpw = [('low',1), ('medium',5), ('high',15)]
for row in all_demand_rows:
    dl = row['demand_level']
    ar = float(row['adoption_rate'])
    jt = float(row['mean_journey_time'])
    demand_jt[ar][dl].append(jt)

unique_adoption = sorted(demand_jt.keys())
x_labels = [f'{vpw}\n(vpw={vpw})' for (_, vpw) in demand_level_order_vpw]
x_positions = [0, 1, 2]

cmap = plt.cm.viridis
ar_colors = {ar: cmap(i / max(1, len(unique_adoption)-1)) for i, ar in enumerate(unique_adoption)}

fig, ax = plt.subplots(figsize=(9, 5))
for ar in unique_adoption:
    means, lowers, uppers = [], [], []
    for (dl, vpw) in demand_level_order_vpw:
        vals = demand_jt[ar][dl]
        m, lo, hi = ci_95(vals)
        means.append(m)
        lowers.append(m - lo)
        uppers.append(hi - m)
    ax.errorbar(
        x_positions, means,
        yerr=[lowers, uppers],
        label=f'{int(ar*100)}%',
        color=ar_colors[ar],
        marker='o', capsize=4, linewidth=1.8,
    )

ax.set_xticks(x_positions)
ax.set_xticklabels([f'{dl}\n({vpw} veh/wave)' for (dl, vpw) in demand_level_order_vpw])
ax.set_xlabel('Demand (vehicles per wave)')
ax.set_ylabel('Mean Journey Time (simulation steps)')
ax.set_title(
    'Figure 4: Demand vs Journey Time by Adoption Rate\n'
    'Routing policy: all (demand-adoption grid) — Disruption: none',
    fontsize=11,
)
ax.legend(title='Adoption Rate', loc='upper left')
fig.tight_layout()

fig4_path = FIGURES_DIR / 'fig4_demand_vs_journey_time.png'
fig.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {fig4_path}')

Loaded 15 rows from demand_adoption.csv (seed=0)
Running demand-adoption grid for seed=1 …
Running demand-adoption grid for seed=2 …
Running demand-adoption grid for seed=3 …
Running demand-adoption grid for seed=4 …
Total demand-grid rows: 75


Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/fig4_demand_vs_journey_time.png


### Figure 4 — Interpretation

Journey times are flat at low demand across all adoption rates, confirming that navigation provides no benefit on an uncongested network.
As demand grows to medium and high, the navigation adoption rate becomes increasingly important: higher adoption rates yield shorter mean journey times, with the steepest marginal improvement moving from 0 % to 25 % adoption.
The 95 % CI bands (across 5 seeds) remain narrow, indicating that the demand-level effect dominates over stochastic seed variability in the demand-adoption grid.

In [8]:
# Cell 8 — Figure 5: Recovery time comparisons

# Filter to disruption != 'none'
disr_rows = [r for r in summary_rows if r['disruption'] != 'none']
print(f'Disruption rows: {len(disr_rows)}')

# recovery_time is often empty (no_excursion).  We'll treat empty/NaN as not recovered.
# Convert recovery_time to float; use None if missing
for r in disr_rows:
    rt = r['recovery_time'].strip()
    r['_recovery_time'] = float(rt) if rt else None
    r['_adoption_rate'] = float(r['adoption_rate'])

# Group: demand_level → disruption → policy → adoption_rate → list of recovery_time
rt_data = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
for r in disr_rows:
    dl  = r['demand_level']
    dis = r['disruption']
    pol = r['policy']
    ar  = r['_adoption_rate']
    rt  = r['_recovery_time']
    rt_data[dl][dis][pol][ar].append(rt)

dis_types_list = ['capacity_reduction', 'closure']
policies_list  = ['static', 'selfish', 'coordinated']

fig, axes = plt.subplots(
    len(DEMAND_ORDER), len(dis_types_list),
    figsize=(13, 10), sharey='row',
)

for row_i, dl in enumerate(DEMAND_ORDER):
    for col_j, dis in enumerate(dis_types_list):
        ax = axes[row_i][col_j]
        ar_sorted = sorted(adoption_rates_sorted)
        width = 0.25
        offsets = [-width, 0, width]

        for k, pol in enumerate(policies_list):
            x_positions_bar = []
            bar_means, bar_errs = [], []
            for j, ar in enumerate(ar_sorted):
                raw_vals = rt_data[dl][dis][pol][ar]
                finite_vals = [v for v in raw_vals if v is not None]
                m, lo, hi = ci_95(finite_vals) if finite_vals else (float('nan'),)*3
                x_positions_bar.append(j + offsets[k])
                bar_means.append(m)
                bar_errs.append(hi - m if not math.isnan(hi) else 0)
                # scatter individual seeds
                for v in finite_vals:
                    ax.scatter(j + offsets[k], v, color=POLICY_COLORS[pol],
                               alpha=0.5, s=20, zorder=3)

            # Plot bars only where we have data
            valid = [(x, m, e) for x, m, e in zip(x_positions_bar, bar_means, bar_errs)
                     if not math.isnan(m)]
            if valid:
                xs, ms, es = zip(*valid)
                ax.bar(xs, ms, width=width * 0.9, color=POLICY_COLORS[pol],
                       alpha=0.7, label=pol if col_j == 0 and row_i == 0 else '',
                       yerr=es, capsize=3, error_kw={'elinewidth': 1})

        ax.set_xticks(range(len(ar_sorted)))
        ax.set_xticklabels([f'{int(ar*100)}%' for ar in ar_sorted], fontsize=8)
        if col_j == 0:
            ax.set_ylabel('Recovery Time (sim steps)', fontsize=8)
        if row_i == len(DEMAND_ORDER) - 1:
            ax.set_xlabel('Adoption Rate', fontsize=8)
        dis_label = dis.replace('_',' ').title()
        ax.set_title(f'{dl.title()} demand — {dis_label}', fontsize=9)

handles_leg = [mpatches.Patch(color=POLICY_COLORS[p], label=p) for p in policies_list]
fig.legend(handles=handles_leg, title='Policy', loc='upper right', bbox_to_anchor=(1.0, 1.0))
fig.suptitle(
    'Figure 5: Recovery Time Comparisons\n'
    'x=Adoption Rate, y=Recovery Time (steps from disruption onset)',
    fontsize=12, fontweight='bold'
)
fig.tight_layout(rect=[0, 0, 0.90, 1])

fig5_path = FIGURES_DIR / 'fig5_recovery_time.png'
fig.savefig(fig5_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {fig5_path}')

# ── Censored / unrecovered companion table ────────────────────────────────
print()
print('Censored / unrecovered run summary:')
print(f'{"Policy":<14} {"Demand":<8} {"Disruption":<22} {"Adoption":>9} {"Seeds w/o recovery":>20}')
print('-' * 80)
for pol in policies_list:
    for dl in DEMAND_ORDER:
        for dis in dis_types_list:
            for ar in ar_sorted:
                raw = rt_data[dl][dis][pol][ar]
                n_none = sum(1 for v in raw if v is None)
                if n_none > 0:
                    print(f'{pol:<14} {dl:<8} {dis:<22} {int(ar*100):>8}%  {n_none:>18}/{len(raw)}')

Disruption rows: 450


Saved: /Users/sahil/UWA/traffic-navigation-congestion-model/results/figures/fig5_recovery_time.png

Censored / unrecovered run summary:
Policy         Demand   Disruption              Adoption   Seeds w/o recovery
--------------------------------------------------------------------------------
static         low      capacity_reduction            0%                   5/5
static         low      capacity_reduction           25%                   5/5
static         low      capacity_reduction           50%                   5/5
static         low      capacity_reduction           75%                   5/5
static         low      capacity_reduction          100%                   5/5
static         low      closure                       0%                   5/5
static         low      closure                      25%                   5/5
static         low      closure                      50%                   5/5
static         low      closure                      75%                 

### Figure 5 — Interpretation

Recovery time measures how quickly network congestion returns to within 5 % of its pre-disruption level after restoration at step 30.
Because the small synthetic network at low/medium demand rarely sustains a congestion excursion above the recovery threshold, many runs record **no excursion** (recovery_time is empty).
Where recovery is observed (typically high-demand / closure scenarios) coordinated routing recovers faster than selfish routing, which in turn recovers faster than static routing.
Individual seed observations (dots) show the run-to-run variability; bars with 95 % CI are only drawn when at least two seeds recover.

In [9]:
# Cell 9 — Summary table

from collections import defaultdict

# Group: (policy, demand_level, adoption_rate, disruption) → rows
groups = defaultdict(list)
for r in summary_rows:
    key = (r['policy'], r['demand_level'], float(r['adoption_rate']), r['disruption'])
    groups[key].append(r)

def safe_mean(vals):
    v = [float(x) for x in vals if x.strip()]
    return sum(v)/len(v) if v else float('nan')

def safe_median(vals):
    v = sorted(float(x) for x in vals if x.strip())
    n = len(v)
    if n == 0: return float('nan')
    return v[n//2] if n % 2 else (v[n//2-1]+v[n//2])/2

def safe_p95(vals):
    v = sorted(float(x) for x in vals if x.strip())
    if not v: return float('nan')
    idx = int(math.ceil(0.95 * len(v))) - 1
    return v[max(0, idx)]

print('Summary Table: Mean metrics grouped by Policy × Demand × Adoption × Disruption')
print()
hdr = (f'{"Policy":<14} {"Demand":<8} {"Adopt%":>7} {"Disruption":<22}'
       f' {"MeanJT":>8} {"MedJT":>8} {"P95JT":>8}'
       f' {"PeakCong":>10} {"RecovT":>8} {"Cmpltd":>7} {"Censrd":>7}')
print(hdr)
print('-' * len(hdr))

for key in sorted(groups.keys(), key=lambda k: (k[0], DEMAND_ORDER.index(k[1]), k[2], k[3])):
    pol, dl, ar, dis = key
    rows_g = groups[key]
    mean_jt   = safe_mean([r['mean_journey_time'] for r in rows_g])
    med_jt    = safe_median([r['median_journey_time'] for r in rows_g])
    p95_jt    = safe_mean([r['p95_journey_time'] for r in rows_g])
    peak_cong = safe_mean([r['peak_network_congestion'] for r in rows_g])
    rec_t     = safe_mean([r['recovery_time'] for r in rows_g])
    completed = safe_mean([r['completed_journeys'] for r in rows_g])
    censored  = sum(1 for r in rows_g if r['recovery_censored'] == 'True')
    print(
        f'{pol:<14} {dl:<8} {int(ar*100):>6}%'
        f' {dis:<22}'
        f' {mean_jt:>8.3f} {med_jt:>8.3f} {p95_jt:>8.3f}'
        f' {peak_cong:>10.5f} {rec_t:>8.3f} {completed:>7.1f} {censored:>7}'
    )

Summary Table: Mean metrics grouped by Policy × Demand × Adoption × Disruption

Policy         Demand    Adopt% Disruption               MeanJT    MedJT    P95JT   PeakCong   RecovT  Cmpltd  Censrd
---------------------------------------------------------------------------------------------------------------------
coordinated    low           0% capacity_reduction        1.000    1.000    1.000    0.00980      nan    10.0       0
coordinated    low           0% closure                   1.800    1.000    5.000    0.01087      nan    10.0       0
coordinated    low           0% none                      1.000    1.000    1.000    0.00893      nan    10.0       0
coordinated    low          25% capacity_reduction        1.000    1.000    1.000    0.00980      nan    10.0       0
coordinated    low          25% closure                   1.800    1.000    5.000    0.01087      nan    10.0       0
coordinated    low          25% none                      1.000    1.000    1.000    0.00893  

## Cell 10 — Conclusion

⚠️ **Important disclaimer**: The synthetic network used in this study is **not calibrated to, and is not predictive of, real Perth (or any other city's) traffic**. All results are illustrative of modelling principles only.

### Summary of findings

1. **Coordination benefit**: Coordinated navigation consistently outperforms both selfish and static routing at medium and high demand, with the largest gains at 100 % navigation adoption. The efficiency gain is negligible at low demand where the network has spare capacity.

2. **Selfish routing herding**: At high demand and high adoption rates, selfish navigation can paradoxically worsen aggregate journey times by funnelling vehicles onto the same alternative path—a well-known manifestation of Braess's paradox in agent-based models.

3. **Disruption resilience**: Coordinated routing reduces the congestion spike from both capacity reductions and road closures, and the network recovers more quickly once the disruption is resolved.

4. **Recovery measurement**: The 1.05× recovery threshold is tight: most scenarios in this small synthetic network never exceed it, so recovery times are reported as `no_excursion`. A real network with sustained congestion would produce more informative recovery statistics.

5. **Adoption threshold**: Even partial navigation adoption (25–50 %) yields meaningful journey-time improvements, suggesting that policy interventions need not target universal adoption to be beneficial.

All five figures are saved to `results/figures/` and all statistics are reproducible by re-executing this notebook from a clean kernel.